# Model Training Notebook

This notebook is used to setup, clean and train the cost prediction model

An example row in the dataset.csv looks like:
```CSV
"Description","Mileage","Make","Model","ModelYear","Total"
"REPAIR (PLUG) TIRE ON CAR","37032","DODGE","CARAVAN -GRAND CARAVAN","2001","5"
"CHANGE OIL & FILTER,TOP OFF FLUIDS","73475","TOYOTA","CAMRY FWD","1997","232"
```

In [ ]:
%pip install scikit-learn
%pip install scikit-multilearn
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install seaborn
%pip install scipy
%pip install xgboost

## Data Setup

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('dataset.csv', header = 0)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().any()

In [ ]:
df.Make.unique()

In [ ]:
df.Model.unique()

## Data Cleanup

In [ ]:
# Remove critical columns that have no value
df = df.dropna(subset=['Description', 'Make', 'Model'])

df.isnull().any()

In [ ]:
# Combine duplicate values
df.Make.replace("MERCEDES-BENZ", "MERCEDES", regex = True, inplace = True)
df.Make.replace("MERCEDES BENZ", "MERCEDES", regex = True, inplace = True)
df.Make.replace("MERCADES-BENZ", "MERCEDES", regex = True, inplace = True)
df.Make.replace("MERCADES-BENS", "MERCEDES", regex = True, inplace = True)
df.Make.replace("VW", "VOLKSWAGEN", regex = True, inplace = True)
df.Make.replace("CHEVY", "CHEVROLET", regex = True, inplace = True)
df.Make.replace("GMC LIGHT TRUCK", "GMC", regex = True, inplace = True)
df.Make.replace("GMC - LIGHT TRUCK", "GMC", regex = True, inplace = True)

df.Make.unique()

In [ ]:
# Drop outliers
df.drop(df[(df.Mileage > 550000) | (df.Mileage <= 0)].index, inplace = True)

# Drop default model year
df.drop(df[df.ModelYear <= 1950].index, inplace = True)

# Drop invalid manufacturers
df.drop(df[df.Make == "SIENNA LE"].index, inplace = True)
df.drop(df[df.Make == "TAURUS SE"].index, inplace = True)
df.drop(df[df.Make == "ALTIMA 2.5S"].index, inplace = True)
df.drop(df[df.Make == "RX350"].index, inplace = True)

# Drop invalid prices
df.drop(df[df.Total < 0].index, inplace = True)

In [ ]:
df.shape

## Dataset Visualization

In [ ]:
import seaborn as sns

In [ ]:
sns.heatmap(df.corr(numeric_only = True));

In [ ]:
sns.pairplot(df, diag_kind = 'kde');

## Building the Model

### Model Training

The model is designed to predict the cost of a repair based on the ticket description, make, model, and model year of the vehicle. 

- X = Predictors
- Y = Target

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
model_df = df

X = model_df[["Description", "ModelYear", "Make", "Model"]]
y = model_df["Total"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.4)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(), "Description"),
        ('onehot', OneHotEncoder(handle_unknown="ignore"), ["Make", "Model"]),
        ('scaler', StandardScaler(), ['ModelYear'])
    ],
    remainder="drop"
)

#### Model 1 - Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

model1 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model1.fit(X_train, y_train)

y_pred = model1.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

#### Model 2 - Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression


model2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LogisticRegression())
])

model2.fit(X_train, y_train)

y_pred = model2.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

#### Model 3 - Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model3 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

# Fit model
model3.fit(X_train, y_train)

# Make predictions
predictions = model3.predict(X_test)

# Measure quality
mse = mean_squared_error(y_test, predictions)
print(f"Mean Squared Error: {mse}")

#### Model 4 - XGBRegressor

In [ ]:
from xgboost import XGBRegressor

model4 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor())
])

# Fit model
model4.fit(X_train, y_train)

# Make predictions
predictions = model4.predict(X_test)

# Measure quality
mse = mean_squared_error(y_test, predictions)
print(f"Mean Squared Error: {mse}")

###  Create Model File

Export the selected model and the vectorizer to a file for use in the API.

In [ ]:
import pickle

pkl_filename = "model.pkl"
with open(pkl_filename, 'wb') as file:
    pickle.dump(model4, file)

### Model Usage

Demo of how to use the model to predict the cost of a service.

In [ ]:
# Load from file
with open(pkl_filename, 'rb') as file:
    pickle_model = pickle.load(file)
    file.close()

# Make a prediction
prediction = pickle_model.predict(pd.DataFrame([{
    "Description": "4 TIRES, FEA",
    "ModelYear": 2014,
    "Make": "TOYOTA",
    "Model": "PRIUS"
}]))

print("Estimated price for that repair: $%.2f" % prediction)